In [1]:
import polars as pl

In [12]:
# create project root dir 
from pathlib import Path
project_root = Path.cwd().parent.resolve()




In [13]:
# reade cleaned budget parquet

data_dir = "data/processed"
df = pl.read_parquet(project_root / data_dir / "2027_cleaned.parquet")

# find unique values in type and fiscal_year columns using polars
print("Unique values in 'type' column:")
print(df.select(pl.col("type").unique()).to_series().to_list())
print("\nUnique values in 'fiscal_year' column:")
print(df.select(pl.col("fiscal_year").unique()).to_series().to_list())

Unique values in 'type' column:
['Budget - Working', 'Budget - Actual', None, 'Budget - Appropriation', 'Budget - Governors Allowance']

Unique values in 'fiscal_year' column:
[2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026, 2027]


In [5]:
# select agency code where agency_code = 'R30' 
agency_code_r30 = df.filter(pl.col("agency_code") == "R30").select(pl.col("agency_code"), pl.col("agency_name")).to_series().to_list()


In [14]:
# Check if R30 exists in the combined data
r30_count = df.filter(pl.col("agency_code") == "R30").height
r30_data = df.filter(pl.col("agency_code") == "R30").select(["agency_code", "agency_name"]).unique()

print(f"Total rows with agency_code = 'R30': {r30_count}")
print(f"Unique agency_code='R30' records:")
print(r30_data)

# Also check all unique agency codes
print(f"\nTotal unique agency codes in combined data: {df.select(pl.col('agency_code').unique()).height}")
print(f"Sample agency codes:")
print(df.select(pl.col("agency_code").unique()).head(10))

Total rows with agency_code = 'R30': 69782
Unique agency_code='R30' records:
shape: (1, 2)
┌─────────────┬───────────────────────────────┐
│ agency_code ┆ agency_name                   │
│ ---         ┆ ---                           │
│ str         ┆ str                           │
╞═════════════╪═══════════════════════════════╡
│ R30         ┆ University System of Maryland │
└─────────────┴───────────────────────────────┘

Total unique agency codes in combined data: 86
Sample agency codes:
shape: (10, 1)
┌─────────────┐
│ agency_code │
│ ---         │
│ str         │
╞═════════════╡
│ X00         │
│ D21         │
│ R75         │
│ D50         │
│ J00         │
│ E17         │
│ R11         │
│ C98         │
│ E80         │
│ S50         │
└─────────────┘


In [15]:
# Check if R30 exists in subprograms.csv
subprograms_path = project_root / "data" / "processed" / "subprograms.csv"
subprograms_df = pl.read_csv(subprograms_path)

print(f"Total rows in subprograms.csv: {subprograms_df.height}")
print(f"Columns: {subprograms_df.columns}")

# Check for R30
r30_in_subprograms = subprograms_df.filter(pl.col("agency_code") == "R30")
print(f"\nRows with agency_code='R30' in subprograms: {r30_in_subprograms.height}")
if r30_in_subprograms.height > 0:
    print(r30_in_subprograms.select(["organization_sub_code", "agency_code", "agency_name", "subprogram_name"]))
else:
    print("R30 NOT FOUND in subprograms!")
    
# Show all unique agency codes in subprograms
print(f"\nTotal unique agency codes in subprograms: {subprograms_df.select(pl.col('agency_code').unique()).height}")
print("Sample agency codes from subprograms:")
print(subprograms_df.select(pl.col("agency_code").unique()).head(15))

Total rows in subprograms.csv: 5542
Columns: ['organization_sub_code', 'organization_code', 'agency_code', 'agency_name', 'unit_code', 'unit_name', 'program_code', 'program_name', 'subprogram_code', 'subprogram_name', 'description', 'category', 'category_title', 'is_it', 'it_designation', 'shadow_it_reason', 'tower', 'sub_tower', 'confidence']

Rows with agency_code='R30' in subprograms: 114
shape: (114, 4)
┌───────────────────────┬─────────────┬──────────────────────┬─────────────────────────────────┐
│ organization_sub_code ┆ agency_code ┆ agency_name          ┆ subprogram_name                 │
│ ---                   ┆ ---         ┆ ---                  ┆ ---                             │
│ str                   ┆ str         ┆ str                  ┆ str                             │
╞═══════════════════════╪═════════════╪══════════════════════╪═════════════════════════════════╡
│ null                  ┆ R30         ┆ University System of ┆ Instruction                     │
│      

In [16]:
# Debug: trace R30 through the dedup process
print("=== DEBUGGING R30 LOSS ===\n")

# 1. Check R30 in original cleaned data
r30_in_cleaned = df.filter(pl.col("agency_code") == "R30")
print(f"1. R30 in 2027_cleaned.parquet: {r30_in_cleaned.height} rows")

# 2. Check organization_sub_code for R30 rows
print(f"2. organization_sub_code distribution for R30:")
r30_org_codes = r30_in_cleaned.select("organization_sub_code").unique()
print(f"   Unique organization_sub_code values: {r30_org_codes.height}")
print(f"   Sample values:")
print(r30_org_codes.head(10))

# 3. Check for nulls in key dedup column
r30_with_nulls = r30_in_cleaned.filter(pl.col("organization_sub_code").is_null())
print(f"\n3. R30 rows with NULL organization_sub_code: {r30_with_nulls.height}")

# 4. Check what happens when we apply dedup logic
print(f"\n4. After dedup by organization_sub_code (latest fiscal year):")
r30_deduped = (
    r30_in_cleaned
    .select(["organization_sub_code", "agency_code", "agency_name", "fiscal_year"])
    .with_columns(pl.col("fiscal_year").cast(pl.Int32, strict=False).alias("_fy_sort"))
    .sort(by=["_fy_sort"], descending=[True], nulls_last=True)
    .unique(subset=["organization_sub_code"], keep="first")
)
print(f"   Result: {r30_deduped.height} unique organization_sub_code values")
print(r30_deduped)

=== DEBUGGING R30 LOSS ===

1. R30 in 2027_cleaned.parquet: 69782 rows
2. organization_sub_code distribution for R30:
   Unique organization_sub_code values: 1
   Sample values:
shape: (1, 1)
┌───────────────────────┐
│ organization_sub_code │
│ ---                   │
│ str                   │
╞═══════════════════════╡
│ null                  │
└───────────────────────┘

3. R30 rows with NULL organization_sub_code: 69782

4. After dedup by organization_sub_code (latest fiscal year):
   Result: 1 unique organization_sub_code values
shape: (1, 5)
┌───────────────────────┬─────────────┬───────────────────────────────┬─────────────┬──────────┐
│ organization_sub_code ┆ agency_code ┆ agency_name                   ┆ fiscal_year ┆ _fy_sort │
│ ---                   ┆ ---         ┆ ---                           ┆ ---         ┆ ---      │
│ str                   ┆ str         ┆ str                           ┆ i32         ┆ i32      │
╞═══════════════════════╪═════════════╪═════════════════════

In [10]:
# Check organization_code for R30 (as fallback for NULL organization_sub_code)
print("=== R30 ORGANIZATION_CODE AS FALLBACK ===\n")

r30_org_code = r30_in_cleaned.select("organization_code").unique()
print(f"Unique organization_code for R30: {r30_org_code.height}")
print(f"Values:")
print(r30_org_code)

# Test: if we use organization_code instead of organization_sub_code
print(f"\n--- If we dedup by organization_code instead ---")
r30_deduped_by_org_code = (
    r30_in_cleaned
    .select(["organization_code", "agency_code", "agency_name", "fiscal_year"])
    .with_columns(pl.col("fiscal_year").cast(pl.Int32, strict=False).alias("_fy_sort"))
    .sort(by=["_fy_sort"], descending=[True], nulls_last=True)
    .unique(subset=["organization_code"], keep="first")
)
print(f"Result: {r30_deduped_by_org_code.height} rows after dedup by organization_code")
print(r30_deduped_by_org_code)

=== R30 ORGANIZATION_CODE AS FALLBACK ===

Unique organization_code for R30: 114
Values:
shape: (114, 1)
┌───────────────────┐
│ organization_code │
│ ---               │
│ str               │
╞═══════════════════╡
│ R30_B23_04        │
│ R30_B27_17        │
│ R30_B37_05        │
│ R30_B30_17        │
│ R30_B37_02        │
│ …                 │
│ R30_B30_07        │
│ R30_B22_17        │
│ R30_B26_06        │
│ R30_B37_17        │
│ R30_B22_02        │
└───────────────────┘

--- If we dedup by organization_code instead ---
Result: 114 rows after dedup by organization_code
shape: (114, 5)
┌───────────────────┬─────────────┬───────────────────────────────┬─────────────┬──────────┐
│ organization_code ┆ agency_code ┆ agency_name                   ┆ fiscal_year ┆ _fy_sort │
│ ---               ┆ ---         ┆ ---                           ┆ ---         ┆ ---      │
│ str               ┆ str         ┆ str                           ┆ i32         ┆ i32      │
╞═══════════════════╪════════════

In [15]:
dim_df = df = pl.read_parquet(project_root / data_dir / "budget_dim.parquet")

# # export this as csv

dim_df.write_csv(project_root / data_dir /"budget_dim.csv")

In [19]:
# load cost pool mappings YAML and join to subobject codes by code
import yaml

mapping_path = project_root / "configs" / "cost_pool_mappings.yaml"

with open(mapping_path, "r", encoding="utf-8") as f:
    mapping_cfg = yaml.safe_load(f) or {}

lookup = mapping_cfg.get("mappings", {})

subobject_df = pl.read_csv(project_root / "data" / "processed" / "subobjects.csv")

# normalize join key to 4-digit code so values like 869 match YAML key 0869
subobject_df = subobject_df.with_columns(
    pl.col("comptroller_subobject_code").cast(pl.Utf8).str.strip_chars().str.zfill(4)
 )

mapping_rows = [
    {
        "comptroller_subobject_code": str(code).strip().zfill(4),
        "cost_pool": attrs.get("cost_pool"),
        "cost_sub_pool": attrs.get("cost_sub_pool"),
    }
    for code, attrs in lookup.items()
    if isinstance(attrs, dict)
]

mapping_df = pl.DataFrame(mapping_rows)

joined_df = subobject_df.join(mapping_df, on="comptroller_subobject_code", how="left")

print("Rows:", joined_df.height)
print("Mapped rows:", joined_df.filter(pl.col("cost_pool").is_not_null()).height)
print("Unmapped rows:", joined_df.filter(pl.col("cost_pool").is_null()).height)

joined_df.select(
    [   "object_code",
        "object_name",
        "comptroller_subobject_code",
        "comptroller_subobject_name",
        "cost_pool",
        "cost_sub_pool",
    ]
).head(20)

Rows: 371
Mapped rows: 371
Unmapped rows: 0


object_code,object_name,comptroller_subobject_code,comptroller_subobject_name,cost_pool,cost_sub_pool
i64,str,str,str,str,str
1,"""Salaries, Wages and Fringe Ben…","""0101""","""Regular Earnings""","""Staffing""","""Internal Labor"""
1,"""Salaries, Wages and Fringe Ben…","""0102""","""Additional Assistance""","""Staffing""","""Internal Labor"""
1,"""Salaries, Wages and Fringe Ben…","""0104""","""Overtime Earnings""","""Staffing""","""Internal Labor"""
1,"""Salaries, Wages and Fringe Ben…","""0105""","""Shift Differential""","""Staffing""","""Internal Labor"""
1,"""Salaries, Wages and Fringe Ben…","""0110""","""Miscellaneous Adjustments""","""Staffing""","""Internal Labor"""
…,…,…,…,…,…
1,"""Salaries, Wages and Fringe Ben…","""0161""","""Employees' Retirement""","""Staffing""","""Internal Labor"""
1,"""Salaries, Wages and Fringe Ben…","""0162""","""Employees' Pension System""","""Staffing""","""Internal Labor"""
1,"""Salaries, Wages and Fringe Ben…","""0163""","""Teachers' Retirement System""","""Staffing""","""Internal Labor"""


In [20]:
# save csv in data/processed
joined_df.write_csv(project_root / data_dir / "subobjects.csv")

In [21]:
# join subprogram with tower classifications on organization_sub_code
subprogram_path = project_root / "data" / "processed" / "subprograms.csv"
tower_cls_path = project_root / "data" / "output" / "tower_classifications.csv"

subprogram_df = pl.read_csv(subprogram_path)
tower_df = pl.read_csv(tower_cls_path, encoding="utf8-lossy")

join_key = "organization_sub_code"

subprogram_df = subprogram_df.with_columns(
    pl.col(join_key).cast(pl.Utf8).str.strip_chars()
)
tower_df = tower_df.with_columns(
    pl.col(join_key).cast(pl.Utf8).str.strip_chars()
)

# keep only join key + last 4 columns from tower classifications
col_needed = tower_df.columns[-4:]
tower_keep_cols = [join_key, *col_needed]

joined_subprogram_tower_df = subprogram_df.join(
    tower_df.select(tower_keep_cols),
    on=join_key,
    how="left",
)

print("Rows:", joined_subprogram_tower_df.height)
print("Tower columns appended:", col_needed)

joined_subprogram_tower_df.head(20)


Rows: 4978
Tower columns appended: ['it_designation', 'tower', 'sub_tower', 'confidence']


organization_sub_code,organization_code,agency_code,agency_name,unit_code,unit_name,program_code,program_name,subprogram_code,subprogram_name,description,category,category_title,is_it,it_designation,shadow_it_reason,it_designation_right,tower,sub_tower,confidence
str,str,str,str,str,str,i64,str,str,str,str,i64,str,bool,str,str,str,str,str,f64
"""A15_O00_01_1BSL""","""A15_O00_01""","""A15""","""Payments to Civil Divisions of…","""O00""","""Payments to Civil Divisions of…",1,"""Disparity Grants""","""1BSL""","""Disparity Grants""","""Section 16-501 of the Local Go…",10,"""Other""",false,null,null,null,null,null,null
"""A15_O00_02_2BSL""","""A15_O00_02""","""A15""","""Payments to Civil Divisions of…","""O00""","""Payments to Civil Divisions of…",2,"""Teacher Retirement Supplementa…","""2BSL""","""Teacher Retirement Supplementa…","""Section 16-503 of the Local Go…",10,"""Other""",false,null,null,null,null,null,null
"""A15_O00_03_3BSL""","""A15_O00_03""","""A15""","""Payments to Civil Divisions of…","""O00""","""Payments to Civil Divisions of…",3,"""Admissions and Amusement Tax D…","""3BSL""","""Admissions and Amusement Tax D…","""The grants in this program rep…",10,"""Other""",false,null,null,null,null,null,null
"""A15_O00_05_5BSL""","""A15_O00_05""","""A15""","""Payments to Civil Divisions of…","""O00""","""Payments to Civil Divisions of…",5,"""Cannabis Sales Tax Distributio…","""5BSL""","""Cannabis Sales Tax Distributio…","""This program represents revenu…",10,"""Other""",false,null,null,null,null,null,null
"""B75_A01_01_0000""","""B75_A01_01""","""B75""","""Legislative Branch""","""A01""","""Legislative Branch""",1,"""Senate""","""0""","""Senate""","""The Senate is composed of 47 S…",8,"""Legislative, Judicial, Legal""",false,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""C00_A00_03_C003""","""C00_A00_03""","""C00""","""Judiciary""","""A00""","""Judiciary""",3,"""Circuit Court Judges""","""C003""","""Circuit Court Masters Child Su…","""The Circuit Courts for Marylan…",8,"""Legislative, Judicial, Legal""",false,null,null,null,null,null,null
"""C00_A00_03_D003""","""C00_A00_03""","""C00""","""Judiciary""","""A00""","""Judiciary""",3,"""Circuit Court Judges""","""D003""","""Law Clerks""","""The Circuit Courts for Marylan…",8,"""Legislative, Judicial, Legal""",false,null,null,null,null,null,null
"""C00_A00_04_B004""","""C00_A00_04""","""C00""","""Judiciary""","""A00""","""Judiciary""",4,"""District Court""","""B004""","""District Court""","""Article IV, Section 1, of the …",8,"""Legislative, Judicial, Legal""",false,null,null,null,null,null,null


In [18]:
# Verify R30 persists through complete pipeline
print("=== R30 FINAL VERIFICATION ===\n")

# 1. Check R30 in final enriched output
final_enriched_path = project_root / "data" / "output" / "final_budget_enriched.parquet"
final_df = pl.read_parquet(final_enriched_path)

r30_final = final_df.filter(pl.col("agency_code") == "R30")
print(f"R30 in final_budget_enriched: {r30_final.height} rows")

# 2. Show sample R30 records with key columns
if r30_final.height > 0:
    print(f"\nSample R30 records from final output:")
    print(r30_final.select([
        "fiscal_year", "agency_code", "agency_name", 
        "organization_code", "organization_sub_code",
        "subprogram_name", "budget", "type"
    ]).head(5))
    
    # 3. Show summary stats
    print(f"\nR30 Summary:")
    print(f"  Total budget (all types): ${r30_final.select('budget').sum().item():,.0f}")
    print(f"  Budget types: {r30_final.select(pl.col('type').unique()).height} unique")
    print(f"  Subprograms: {r30_final.select(pl.col('subprogram_name').unique()).height} unique")
else:
    print("⚠️ R30 NOT FOUND in final output!")

=== R30 FINAL VERIFICATION ===

R30 in final_budget_enriched: 0 rows
⚠️ R30 NOT FOUND in final output!


In [20]:
# Debug: trace R30 through downstream stages (simplified)
print("=== TRACING R30 THROUGH DOWNSTREAM STAGES ===\n")

# 1. Check cleaned data
cleaned_path = project_root / "data" / "processed" / "2027_cleaned.parquet"
cleaned_df = pl.read_parquet(cleaned_path)
r30_cleaned = cleaned_df.filter(pl.col("agency_code") == "R30")
print(f"1. R30 in 2027_cleaned.parquet: {r30_cleaned.height} rows")

# 2. Check subprograms.csv
subprograms_path = project_root / "data" / "processed" / "subprograms.csv"
subprograms_df = pl.read_csv(subprograms_path)
r30_subprograms = subprograms_df.filter(pl.col("agency_code") == "R30")
print(f"2. R30 in subprograms.csv: {r30_subprograms.height} rows")

# 3. Check if NULL organization_sub_code causes join issue
print(f"\n3. NULL join issue test:")
r30_budget_subset = r30_cleaned.select(["organization_sub_code"]).unique()
r30_subprograms_subset = r30_subprograms.select(["organization_sub_code"]).unique()

print(f"   Budget NULL org_sub_codes: {r30_budget_subset.filter(pl.col('organization_sub_code').is_null()).height}")
print(f"   Subprograms NULL org_sub_codes: {r30_subprograms_subset.filter(pl.col('organization_sub_code').is_null()).height}")

# Simulate LEFT join on NULL
test_budget = r30_budget_subset.with_columns(pl.lit("budget_row").alias("source"))
test_subprograms = r30_subprograms_subset.with_columns(pl.lit("subprograms_row").alias("source"))

joined = test_budget.join(test_subprograms, on="organization_sub_code", how="left", suffix="_subprog")
print(f"\n   After LEFT join: {joined.height} rows")
print(f"   Rows with NULL source from subprograms: {joined.filter(pl.col('source_subprog').is_null()).height}")
print(f"   This shows NULL values DON'T match in joins")

=== TRACING R30 THROUGH DOWNSTREAM STAGES ===

1. R30 in 2027_cleaned.parquet: 69782 rows
2. R30 in subprograms.csv: 114 rows

3. NULL join issue test:
   Budget NULL org_sub_codes: 1
   Subprograms NULL org_sub_codes: 1

   After LEFT join: 1 rows
   Rows with NULL source from subprograms: 1
   This shows NULL values DON'T match in joins


In [23]:
# Check is_it status and tower for R30 in subprograms
print("\n=== R30 IT CLASSIFICATION ===\n")

# Count by is_it
is_it_counts = r30_subprograms.group_by("is_it").agg(pl.len().alias("count"))
print(f"R30 rows by is_it status:")
print(is_it_counts)

# Check tower values
print(f"\nUnique tower values for R30: {r30_subprograms.select('tower').unique().height}")
tower_counts = r30_subprograms.group_by("tower").agg(pl.len().alias("count")).sort("count", descending=True)
print(tower_counts)


=== R30 IT CLASSIFICATION ===

R30 rows by is_it status:
shape: (1, 2)
┌───────┬───────┐
│ is_it ┆ count │
│ ---   ┆ ---   │
│ bool  ┆ u32   │
╞═══════╪═══════╡
│ true  ┆ 114   │
└───────┴───────┘

Unique tower values for R30: 1
shape: (1, 2)
┌───────┬───────┐
│ tower ┆ count │
│ ---   ┆ ---   │
│ str   ┆ u32   │
╞═══════╪═══════╡
│ null  ┆ 114   │
└───────┴───────┘


In [24]:
# Test: simulate build_final_enriched join for R30
print("\n=== SIMULATING BUILD_FINAL_ENRICHED JOIN ===\n")

# Load all data (not just R30 sample)
budget_df = pl.read_parquet(project_root / "data" / "processed" / "2027_cleaned.parquet")
subprograms_full = pl.read_csv(project_root / "data" / "processed" / "subprograms.csv")

# Check total counts before join
print(f"Budget total rows: {budget_df.height}")
print(f"Subprograms total rows: {subprograms_full.height}")

# Normalize keys as build_final_enriched does
budget_df = budget_df.with_columns(pl.col("organization_sub_code").cast(pl.Utf8))
subprograms_full = subprograms_full.with_columns(pl.col("organization_sub_code").cast(pl.Utf8))

# LEFT join as build_final_enriched does
merged = budget_df.join(subprograms_full, on="organization_sub_code", how="left")
print(f"\nAfter LEFT join: {merged.height} rows")
print(f"Rows where 'is_it' from subprograms is not null: {merged.filter(pl.col('is_it').is_not_null()).height}")

# Check R30 after join
r30_merged = merged.filter(pl.col("agency_code") == "R30")
print(f"\nR30 in merged result: {r30_merged.height} rows")


=== SIMULATING BUILD_FINAL_ENRICHED JOIN ===

Budget total rows: 826239
Subprograms total rows: 5542

After LEFT join: 826239 rows
Rows where 'is_it' from subprograms is not null: 741793

R30 in merged result: 69782 rows


In [25]:
# Continue simulation: add subobjects join
print("\n=== CONTINUING SIMULATION WITH SUBOBJECTS JOIN ===\n")

# Load subobject_codes
subobject_codes_path = project_root / "data" / "processed" / "subobject_codes.csv"
subobjects = pl.read_csv(subobject_codes_path)

print(f"Subobjects total rows: {subobjects.height}")

# Simulate second join on comptroller_subobject_code
merged2 = merged.with_columns(
    pl.col("comptroller_subobject_code").cast(pl.Utf8).str.pad_start(4, "0")
)
subobjects = subobjects.with_columns(
    pl.col("comptroller_subobject_code").cast(pl.Utf8).str.pad_start(4, "0")
)

merged_final = merged2.join(subobjects, on="comptroller_subobject_code", how="left")

print(f"After subobjects LEFT join: {merged_final.height} rows")
print(f"R30 after subobjects join: {merged_final.filter(pl.col('agency_code') == 'R30').height} rows")

# Compare with actual final output
print(f"\nActual final_budget_enriched rows: {final_df.height}")
print(f"Our simulation rows: {merged_final.height}")


=== CONTINUING SIMULATION WITH SUBOBJECTS JOIN ===

Subobjects total rows: 387
After subobjects LEFT join: 826239 rows
R30 after subobjects join: 69782 rows

Actual final_budget_enriched rows: 826239
Our simulation rows: 826239


In [26]:
# Check if tower_classifications.csv has R30
print("\n=== CHECKING TOWER CLASSIFICATIONS ===\n")

tower_class_path = project_root / "data" / "output" / "tower_classifications.csv"
if tower_class_path.exists():
    tower_class_df = pl.read_csv(tower_class_path)
    print(f"tower_classifications.csv rows: {tower_class_df.height}")
    print(f"Columns: {tower_class_df.columns}")
    
    r30_tower = tower_class_df.filter(pl.col("organization_sub_code").is_null() | (pl.col("organization_sub_code") == ""))
    print(f"\nRows with NULL organization_sub_code: {r30_tower.height}")
    
    if r30_tower.height > 0:
        print(f"Sample NULL org_sub_code rows from tower_classifications:")
        print(r30_tower.head(3))
else:
    print(f"tower_classifications.csv does NOT exist!")


=== CHECKING TOWER CLASSIFICATIONS ===

tower_classifications.csv rows: 456
Columns: ['organization_sub_code', 'subprogram_code', 'subprogram_name', 'agency_name', 'it_designation', 'tower', 'sub_tower', 'confidence']

Rows with NULL organization_sub_code: 0


In [28]:
# Check IT designations config directly
print("\n=== CHECKING IT DESIGNATION PATTERNS ===\n")

import yaml

# Read IT programs config
config_path = project_root / "configs" / "it_programs.yaml"
with open(config_path, "r", encoding="utf-8") as f:
    it_config = yaml.safe_load(f) or {}

designations = it_config.get("it_programs", {}).get("designations", [])

print(f"Number of designations configured: {len(designations)}")

# Check if any designation matches R30 or "University System of Maryland"
r30_matched = False
for d in designations:
    patterns = d.get("patterns", [])
    for p in patterns:
        if "R30" in str(p).upper() or "university" in str(p).lower() or "maryland" in str(p).lower():
            print(f"\nFound R30-related match in designation: {d.get('id')}")
            print(f"  Pattern: {p}")
            r30_matched = True

if not r30_matched:
    print(f"\n❌ No explicit R30 or University System of Maryland patterns found in designations")
    print(f"\nThis means R30 will be marked as is_it=FALSE by designation_filter")


=== CHECKING IT DESIGNATION PATTERNS ===

Number of designations configured: 3

❌ No explicit R30 or University System of Maryland patterns found in designations

This means R30 will be marked as is_it=FALSE by designation_filter


In [29]:
# Check CURRENT subprograms.csv after all downstream stages
print("\n=== CHECKING CURRENT SUBPROGRAMS.CSV ===\n")

subprograms_current = pl.read_csv(project_root / "data" / "processed" / "subprograms.csv")
r30_current = subprograms_current.filter(pl.col("agency_code") == "R30")

print(f"Total rows in subprograms.csv now: {subprograms_current.height}")
print(f"R30 rows in subprograms.csv now: {r30_current.height}")

if r30_current.height > 0:
    print(f"\nR30 unique is_it values: {r30_current.select('is_it').unique()}")
    is_it_breakdown = r30_current.group_by("is_it").agg(pl.len().alias("count"))
    print(f"R30 breakdown by is_it:")
    print(is_it_breakdown)
else:
    print(f"\n❌ R30 is COMPLETELY GONE from subprograms.csv")


=== CHECKING CURRENT SUBPROGRAMS.CSV ===

Total rows in subprograms.csv now: 5542
R30 rows in subprograms.csv now: 114

R30 unique is_it values: shape: (1, 1)
┌───────┐
│ is_it │
│ ---   │
│ bool  │
╞═══════╡
│ true  │
└───────┘
R30 breakdown by is_it:
shape: (1, 2)
┌───────┬───────┐
│ is_it ┆ count │
│ ---   ┆ ---   │
│ bool  ┆ u32   │
╞═══════╪═══════╡
│ true  ┆ 114   │
└───────┴───────┘


In [ ]:
# Fresh reload of final_budget_enriched to eliminate stale data
print("\n=== FRESH RELOAD OF FINAL_BUDGET_ENRICHED ===\n")

final_parquet_path = project_root / "data" / "output" / "final_budget_enriched.parquet"
print(f"Loading from: {final_parquet_path}")
print(f"File exists: {final_parquet_path.exists()}")

final_df_fresh = pl.read_parquet(final_parquet_path)

print(f"Total rows: {final_df_fresh.height}")
print(f"Columns: {len(final_df_fresh.columns)}")

r30_final_fresh = final_df_fresh.filter(pl.col("agency_code") == "R30")
print(f"R30 rows: {r30_final_fresh.height}")

# Also check what agency codes ARE in the final output
print(f"\nTotal unique agency codes: {final_df_fresh.select(pl.col('agency_code').unique()).height}")
unique_agencies = final_df_fresh.select(pl.col("agency_code").unique()).sort("agency_code")
print(f"Sample agency codes (first 20):")
print(unique_agencies.head(20))


=== MANUALLY REPRODUCING BUILD_FINAL_ENRICHED JOINS ===

Budget: 826239 rows
Subprograms: 5542 rows
Subobjects: 387 rows

Before joins:
  R30 in budget: 69782
  R30 in subprograms: 114

After subprograms LEFT join:
  Total rows: 826239
  R30 rows: 69782

After subobjects LEFT join (with padding):
  Total rows: 826239
  R30 rows: 69782

Actual final_budget_enriched:
  Total rows: 826239
  R30 rows: 0

Final output file check:


ComputeError: could not parse `090x` as dtype `i64` at column 'agency_subobject_code' (column number 3)

The current offset in the file is 2301994 bytes.

You might want to try:
- increasing `infer_schema_length` (e.g. `infer_schema_length=10000`),
- specifying correct dtype with the `schema_overrides` argument
- setting `ignore_errors` to `True`,
- adding `090x` to the `null_values` list.

Original error: ```remaining bytes non-empty```

In [22]:
joined_subprogram_tower_df.write_csv(project_root / "data" / "processed" / "subprograms.csv")

In [ ]:
# join budget parquet with IT subprograms and subobject mappings

budget_path = project_root / "data" / "processed" / "budget_cleaned.parquet"
budget_df = pl.read_parquet(budget_path)

it_subprograms_path = project_root / "data" / "processed" / "it_subprogram.csv"
it_subprograms_df = pl.read_csv(it_subprograms_path)

subobject_codes_path = project_root / "data" / "processed" / "subobjects.csv"
subobject_codes_df = pl.read_csv(subobject_codes_path)

# normalize join key to 4-digit strings so 869 and 0869 match
budget_df = budget_df.with_columns([
    pl.col("comptroller_subobject_code").cast(pl.Utf8).str.strip_chars().str.zfill(4).alias("comptroller_subobject_code")
])

subobject_codes_df = subobject_codes_df.with_columns([
    pl.col("comptroller_subobject_code").cast(pl.Utf8).str.strip_chars().str.zfill(4).alias("comptroller_subobject_code")
])

budget_columns = ['fiscal_year',
 'organization_sub_code',                 
 'comptroller_subobject_code',               
 'agency_subobject_code',
 'agency_subobject_name',
 'fund_type_name',
 'budget',
 'type',
 'organization_code',
 'category',
 'category_title']

joined_df = budget_df.select(budget_columns
                             ).join(
    it_subprograms_df,
    on="organization_sub_code",
    how="left"
).join(
    subobject_codes_df,
    on="comptroller_subobject_code",
    how="left"
)

In [46]:
joined_df.columns

['fiscal_year',
 'organization_sub_code',
 'comptroller_subobject_code',
 'agency_subobject_code',
 'agency_subobject_name',
 'fund_type_name',
 'budget',
 'type',
 'organization_code',
 'category',
 'category_title',
 'agency_code',
 'agency_name',
 'unit_code',
 'unit_name',
 'program_code',
 'program_name',
 'subprogram_code',
 'subprogram_name',
 'organization_code_right',
 'description',
 'is_IT',
 'IT_designination',
 'tower',
 'sub_tower',
 'confidence',
 'object_code',
 'object_name',
 'comptroller_subobject_name',
 'cost_pool',
 'cost_sub_pool']

In [55]:
joined_df.write_csv(project_root / "data" / "enriched" / "budget_enriched.csv")
joined_df.write_parquet(project_root / "data" / "enriched" / "budget_enriched.parquet")